# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: b8dbec7d-1b9b-4502-bf9d-662ff75c4b25
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session b8dbec7d-1b9b-4502-bf9d-662ff75c4b25 to get into ready status...
Session b8dbec7d-1b9b-4502-bf9d-662ff75c4b25 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
order_items_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='order_items')

order_items_dyf.printSchema()

root
|-- order_id: string
|-- order_item_id: long
|-- product_id: string
|-- seller_id: string
|-- shipping_limit_date: string
|-- price: double
|-- freight_value: double


In [3]:
order_items_dyf = order_items_dyf.apply_mapping([
    ("order_id","string","order_id","string"),
    ("order_item_id","bigint","order_item_id","string"),
    ("product_id","string","product_id","string"),
    ("seller_id","string","seller_id","string"),
    ("shipping_limit_date","string","shipping_limit_date","timestamp"),
    ("price","double","price","double"),
    ("freight_value","double","freight_value","double")
])

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [4]:
# I transform both DynamicFrames into pyspark Dataframes to use pyspark SQl
order_items_df = order_items_dyf.toDF()
order_items_df_sample = order_items_df.limit(1000)

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
# Step 1: Keep only rows with positive prices
cleaned_oi_df = order_items_df_sample.filter(order_items_df["price"] >= 0.0)

# Step 2: Calculate total price for an item, which is price + freight value (shipping costs)
cleaned_oi_df = cleaned_oi_df.withColumn("total_price", cleaned_oi_df["price"] + cleaned_oi_df["freight_value"])

In [6]:
cleaned_oi_df.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|       total_price|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+------------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|             72.19|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|            259.83|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|            216.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|             25.78|
|00042b26cf59d7ce6...|     

In [7]:
# Convert the pyspark Dataframe back into a DynamicFrame
from awsglue.dynamicframe import DynamicFrame

cleaned_dyf = DynamicFrame.fromDF(cleaned_oi_df, glueContext, "order_items_dyf")

In [8]:
# Store the cleaned dataset as Parquet in S3
s3output = glueContext.getSink(
  path="s3://bucket181rt2/clean/order_items",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_clean", catalogTableName="order_items"
)
s3output.setFormat("parquet", useGlueParquetWriter=True)
s3output.writeFrame(cleaned_dyf)